# String Pot Analysis

This notebook is a standalone analysis workspace for the homemade string potentiometer test data. It stays separate from the main `bodaqs_analysis` package and works directly from a logger CSV plus the crank-drive geometry summary in this folder.

Workflow:
- load a CSV and select the time plus raw ADC columns
- apply a configurable cubic correction in corrected-count units
- unwrap the corrected signal with a configurable wrap span
- estimate the cycle period from the measured signal
- build the ideal string displacement profile from the crank geometry
- fit only scale and time shift between the measured and ideal profiles
- inspect residuals and summary metrics

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import optimize, signal

pio.renderers.default = 'notebook_connected'

In [ ]:
# Parameters

analysis_dir = Path('.')
csv_path = analysis_dir / '1000rpm.CSV'

time_column = 'timestamp'
raw_adc_column = 'i2c_stringpot_raw [counts]'
time_format = '%H:%M:%S.%f'
analysis_start_time_s = 0
analysis_end_time_s = 120
wrap_span_counts = 4095.0
wrap_low_band_max_count = 1000.0
wrap_high_band_min_count = 204.0

# Cubic correction in corrected-count units.
c0 = 0.0
c1 = 1
c2 = 0
c3 = 0.0

# Crank-drive geometry in mm.
crank_pin_offset_r_mm = 68.2
entry_offset_l_mm = 450.0

# Period estimation and fitting controls.
sample_rate_hz_override = None
driving_speed_rpm_override = None
period_search_min_rpm = 30.0
period_search_max_rpm = 600.0
fit_shift_search_fraction_of_period = 0.50
minimum_valid_fraction_for_fit = 0.80

# Instantaneous-period model controls.
instantaneous_period_min_peak_spacing_fraction = 0.60
instantaneous_period_peak_prominence_fraction = 0.15
instantaneous_period_smoothing_fraction_of_period = 0.20

In [ ]:
def _require_column(df: pd.DataFrame, column_name: str) -> None:
    if column_name not in df.columns:
        raise KeyError(f"Column '{column_name}' not found. Available columns: {list(df.columns)}")


def load_time_seconds(series: pd.Series, time_format: str | None = None) -> np.ndarray:
    numeric = pd.to_numeric(series, errors='coerce')
    numeric_values = numeric.to_numpy(dtype=float)
    if np.isfinite(numeric_values).any() and np.isfinite(numeric_values).sum() == len(series):
        first_idx = np.flatnonzero(np.isfinite(numeric_values))[0]
        return numeric_values - numeric_values[first_idx]

    parsed = pd.to_datetime(series, format=time_format, errors='coerce')
    valid = parsed.notna().to_numpy(dtype=bool)
    if not valid.any():
        raise ValueError('Could not convert the time column to a relative-seconds vector.')

    base_time = parsed[valid].iloc[0]
    rel_s = (parsed - base_time).dt.total_seconds().to_numpy(dtype=float)
    return rel_s


def apply_polynomial_correction(raw_counts: np.ndarray, coeffs: tuple[float, float, float, float]) -> np.ndarray:
    c0_, c1_, c2_, c3_ = coeffs
    raw_counts = np.asarray(raw_counts, dtype=float)
    return c0_ + c1_ * raw_counts + c2_ * raw_counts**2 + c3_ * raw_counts**3


def unwrap_periodic_signal(
    values: np.ndarray,
    wrap_span: float,
    low_band_max_count: float,
    high_band_min_count: float,
) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    out = values.copy()
    valid_idx = np.flatnonzero(np.isfinite(values))
    if valid_idx.size == 0:
        return out

    if not (0.0 <= low_band_max_count < high_band_min_count <= wrap_span):
        raise ValueError('Wrap bands must satisfy 0 <= low_band_max_count < high_band_min_count <= wrap_span.')

    offset = 0.0
    prev = values[valid_idx[0]]
    out[valid_idx[0]] = prev
    last_direction = 0

    for pos in range(1, valid_idx.size):
        idx = valid_idx[pos]
        current = values[idx]

        if pos >= 2:
            prev_prev = values[valid_idx[pos - 2]]
            previous_delta = prev - prev_prev
            if np.isfinite(previous_delta) and previous_delta > 0.0:
                last_direction = 1
            elif np.isfinite(previous_delta) and previous_delta < 0.0:
                last_direction = -1

        prev_is_low = prev <= low_band_max_count
        prev_is_high = prev >= high_band_min_count
        current_is_low = current <= low_band_max_count
        current_is_high = current >= high_band_min_count

        if last_direction > 0 and prev_is_high and current_is_low:
            offset += wrap_span
        elif last_direction < 0 and prev_is_low and current_is_high:
            offset -= wrap_span

        out[idx] = current + offset
        prev = current

    return out


def _safe_savgol_window(length: int, requested: int, poly_order: int) -> int | None:
    if length <= poly_order:
        return None
    window = min(requested, length if length % 2 == 1 else length - 1)
    if window <= poly_order:
        window = poly_order + 2
        if window % 2 == 0:
            window += 1
    if window > length:
        window = length if length % 2 == 1 else length - 1
    if window <= poly_order or window < 3:
        return None
    return window


def smooth_series(values: np.ndarray, requested_window: int, poly_order: int) -> np.ndarray:
    y = np.asarray(values, dtype=float)
    valid = np.isfinite(y)
    if valid.sum() < poly_order + 2:
        return y.copy()

    filled = pd.Series(y).interpolate(limit_direction='both').to_numpy(dtype=float)
    window = _safe_savgol_window(len(filled), requested_window, poly_order)
    if window is None:
        return filled
    return signal.savgol_filter(filled, window_length=window, polyorder=poly_order, mode='interp')


def estimate_sample_rate_hz(t_s: np.ndarray, sample_rate_hz_override: float | None = None) -> float:
    if sample_rate_hz_override is not None:
        sample_rate_hz = float(sample_rate_hz_override)
        if not np.isfinite(sample_rate_hz) or sample_rate_hz <= 0.0:
            raise ValueError('sample_rate_hz_override must be positive when provided.')
        return sample_rate_hz

    t_s = np.asarray(t_s, dtype=float)
    dt = np.diff(t_s)
    dt = dt[np.isfinite(dt) & (dt > 0.0)]
    if dt.size == 0:
        raise ValueError('Could not infer a positive sample interval from the time vector.')

    dt_median = float(np.median(dt))
    return 1.0 / dt_median


def estimate_period_seconds(
    t_s: np.ndarray,
    wrapped_counts: np.ndarray,
    wrap_span: float,
    min_rpm: float,
    max_rpm: float,
    sample_rate_hz_override: float | None = None,
) -> dict:
    mask = np.isfinite(t_s) & np.isfinite(wrapped_counts)
    if mask.sum() < 10:
        raise ValueError('Need at least 10 valid samples to estimate period.')

    t_valid = np.asarray(t_s[mask], dtype=float)
    y_valid = np.asarray(wrapped_counts[mask], dtype=float)

    order = np.argsort(t_valid)
    t_valid = t_valid[order]
    y_valid = y_valid[order]

    sample_rate_hz = estimate_sample_rate_hz(t_valid, sample_rate_hz_override=sample_rate_hz_override)
    if not np.isfinite(min_rpm) or not np.isfinite(max_rpm) or min_rpm <= 0.0 or max_rpm <= min_rpm:
        raise ValueError('RPM search bounds must be finite and satisfy 0 < min_rpm < max_rpm.')

    phase_rad = 2.0 * np.pi * y_valid / float(wrap_span)
    circular_cos = np.cos(phase_rad)
    circular_sin = np.sin(phase_rad)
    circular_cos_centered = circular_cos - np.mean(circular_cos)
    circular_sin_centered = circular_sin - np.mean(circular_sin)

    autocorr = signal.correlate(circular_cos_centered, circular_cos_centered, mode='full')
    autocorr += signal.correlate(circular_sin_centered, circular_sin_centered, mode='full')
    autocorr = autocorr[autocorr.size // 2 :]
    lag_samples = np.arange(autocorr.size, dtype=float)
    lag_s = lag_samples / sample_rate_hz

    min_period_s = 60.0 / float(max_rpm)
    max_period_s = 60.0 / float(min_rpm)
    search_mask = (lag_s >= min_period_s) & (lag_s <= max_period_s)
    if not np.any(search_mask):
        raise ValueError('RPM search bounds do not map to any valid lag samples.')

    search_idx = np.flatnonzero(search_mask)
    autocorr_search = autocorr[search_idx]
    peak_offsets, _ = signal.find_peaks(autocorr_search)
    if peak_offsets.size == 0:
        best_idx = int(search_idx[np.argmax(autocorr_search)])
        candidate_periods = np.asarray([lag_s[best_idx]], dtype=float)
        candidate_strengths = np.asarray([autocorr[best_idx]], dtype=float)
    else:
        peak_idx = search_idx[peak_offsets]
        peak_strengths = autocorr[peak_idx]
        order = np.argsort(peak_strengths)[::-1]
        peak_idx = peak_idx[order]
        peak_strengths = peak_strengths[order]
        candidate_periods = lag_s[peak_idx]
        candidate_strengths = peak_strengths

    period_s = float(candidate_periods[0])
    if not np.isfinite(period_s) or period_s <= 0.0:
        raise ValueError('Estimated period is not positive.')

    nperseg = min(len(y_valid), 4096)
    if nperseg < 8:
        raise ValueError('Not enough valid samples for spectral cross-check.')
    freq_hz, psd_cos = signal.welch(circular_cos_centered, fs=sample_rate_hz, nperseg=nperseg)
    _, psd_sin = signal.welch(circular_sin_centered, fs=sample_rate_hz, nperseg=nperseg)
    psd_total = psd_cos + psd_sin
    search_freq_mask = (freq_hz >= 1.0 / max_period_s) & (freq_hz <= 1.0 / min_period_s)
    dominant_frequency_hz = float(freq_hz[search_freq_mask][np.argmax(psd_total[search_freq_mask])])

    return {
        'period_s': period_s,
        'method': 'circular-autocorrelation',
        'sample_rate_hz': sample_rate_hz,
        't_valid': t_valid,
        'wrapped_counts_valid': y_valid,
        'circular_cos': circular_cos,
        'circular_sin': circular_sin,
        'autocorr': autocorr,
        'lag_s': lag_s,
        'search_mask': search_mask,
        'candidate_periods_s': np.asarray(candidate_periods, dtype=float),
        'candidate_strengths': np.asarray(candidate_strengths, dtype=float),
        'dominant_frequency_hz': dominant_frequency_hz,
    }


def ideal_string_displacement_mm(theta_rad: np.ndarray, r_mm: float, l_mm: float) -> np.ndarray:
    theta_rad = np.asarray(theta_rad, dtype=float)
    s_mm = np.sqrt(l_mm**2 + r_mm**2 - 2.0 * l_mm * r_mm * np.cos(theta_rad))
    return s_mm - (l_mm - r_mm)


def ideal_profile_from_time(t_s: np.ndarray, period_s: float, r_mm: float, l_mm: float) -> np.ndarray:
    theta = 2.0 * np.pi * np.asarray(t_s, dtype=float) / period_s
    return ideal_string_displacement_mm(theta, r_mm=r_mm, l_mm=l_mm)


def ideal_profile_from_phase(phase_rad: np.ndarray, r_mm: float, l_mm: float) -> np.ndarray:
    return ideal_string_displacement_mm(np.asarray(phase_rad, dtype=float), r_mm=r_mm, l_mm=l_mm)


def phase_from_anchor_times(
    t_s: np.ndarray,
    anchor_times_s: np.ndarray,
    anchor_phase_rad: float = np.pi,
) -> dict:
    anchor_times_s = np.asarray(anchor_times_s, dtype=float)
    anchor_times_s = anchor_times_s[np.isfinite(anchor_times_s)]
    anchor_times_s = np.unique(anchor_times_s)
    if anchor_times_s.size < 2:
        raise ValueError('Need at least two cycle anchors to build a variable-speed phase model.')

    cycle_periods_s = np.diff(anchor_times_s)
    if not np.all(np.isfinite(cycle_periods_s)) or np.any(cycle_periods_s <= 0.0):
        raise ValueError('Cycle anchors must produce strictly positive cycle periods.')

    t = np.asarray(t_s, dtype=float)
    phase_full = np.full(t.shape, np.nan, dtype=float)
    period_full = np.full(t.shape, np.nan, dtype=float)
    valid_idx = np.flatnonzero(np.isfinite(t))
    if valid_idx.size == 0:
        return {
            'phase_rad': phase_full,
            'instantaneous_period_s': period_full,
            'cycle_periods_s': cycle_periods_s,
        }

    t_valid = t[valid_idx]
    first_period_s = float(cycle_periods_s[0])
    last_period_s = float(cycle_periods_s[-1])
    two_pi = 2.0 * np.pi

    before_mask = t_valid < anchor_times_s[0]
    if np.any(before_mask):
        frac = (t_valid[before_mask] - anchor_times_s[0]) / first_period_s
        phase_full[valid_idx[before_mask]] = anchor_phase_rad + two_pi * frac
        period_full[valid_idx[before_mask]] = first_period_s

    for cycle_idx, period_s in enumerate(cycle_periods_s):
        in_cycle = (t_valid >= anchor_times_s[cycle_idx]) & (t_valid < anchor_times_s[cycle_idx + 1])
        if np.any(in_cycle):
            frac = (t_valid[in_cycle] - anchor_times_s[cycle_idx]) / period_s
            phase_full[valid_idx[in_cycle]] = anchor_phase_rad + two_pi * cycle_idx + two_pi * frac
            period_full[valid_idx[in_cycle]] = period_s

    after_mask = t_valid >= anchor_times_s[-1]
    if np.any(after_mask):
        frac = (t_valid[after_mask] - anchor_times_s[-1]) / last_period_s
        phase_full[valid_idx[after_mask]] = anchor_phase_rad + two_pi * (anchor_times_s.size - 1) + two_pi * frac
        period_full[valid_idx[after_mask]] = last_period_s

    return {
        'phase_rad': phase_full,
        'instantaneous_period_s': period_full,
        'cycle_periods_s': cycle_periods_s,
    }


def estimate_instantaneous_period_model(
    t_s: np.ndarray,
    measured_counts_zeroed: np.ndarray,
    nominal_period_s: float,
    sample_rate_hz: float,
    min_peak_spacing_fraction: float = 0.60,
    peak_prominence_fraction: float = 0.15,
    smoothing_fraction_of_period: float = 0.20,
) -> dict:
    mask = np.isfinite(t_s) & np.isfinite(measured_counts_zeroed)
    if mask.sum() < 10:
        raise ValueError('Need at least 10 valid samples to estimate instantaneous periods.')

    t_valid = np.asarray(t_s[mask], dtype=float)
    y_valid = np.asarray(measured_counts_zeroed[mask], dtype=float)
    order = np.argsort(t_valid)
    t_valid = t_valid[order]
    y_valid = y_valid[order]

    if not np.isfinite(nominal_period_s) or nominal_period_s <= 0.0:
        raise ValueError('nominal_period_s must be positive.')
    if not np.isfinite(sample_rate_hz) or sample_rate_hz <= 0.0:
        raise ValueError('sample_rate_hz must be positive.')

    requested_window = int(np.round(max(5.0, smoothing_fraction_of_period * nominal_period_s * sample_rate_hz)))
    if requested_window % 2 == 0:
        requested_window += 1
    smoothed_signal = smooth_series(y_valid, requested_window=requested_window, poly_order=3)

    amplitude = float(np.nanmax(smoothed_signal) - np.nanmin(smoothed_signal))
    prominence = max(1.0, float(peak_prominence_fraction) * amplitude)
    min_distance_samples = max(1, int(np.floor(float(min_peak_spacing_fraction) * nominal_period_s * sample_rate_hz)))

    peak_idx, _ = signal.find_peaks(
        smoothed_signal,
        distance=min_distance_samples,
        prominence=prominence,
    )
    if peak_idx.size < 2:
        raise ValueError('Could not detect enough cycle anchors from the measured signal.')

    anchor_times_s = t_valid[peak_idx]
    anchor_values = smoothed_signal[peak_idx]
    phase_info = phase_from_anchor_times(t_s, anchor_times_s, anchor_phase_rad=np.pi)
    cycle_periods_s = phase_info['cycle_periods_s']
    cycle_mid_times_s = 0.5 * (anchor_times_s[:-1] + anchor_times_s[1:])
    cycle_speed_rpm = 60.0 / cycle_periods_s

    return {
        'anchor_times_s': anchor_times_s,
        'anchor_values': anchor_values,
        'peak_indices': peak_idx,
        'smoothed_signal': smoothed_signal,
        't_valid': t_valid,
        'signal_valid': y_valid,
        'requested_window': requested_window,
        'prominence': prominence,
        'min_distance_samples': min_distance_samples,
        'phase_rad': phase_info['phase_rad'],
        'instantaneous_period_s': phase_info['instantaneous_period_s'],
        'cycle_periods_s': cycle_periods_s,
        'cycle_mid_times_s': cycle_mid_times_s,
        'cycle_speed_rpm': cycle_speed_rpm,
    }


def shift_signal_in_time(t_ref: np.ndarray, t_src: np.ndarray, y_src: np.ndarray, shift_s: float) -> np.ndarray:
    shifted_time = np.asarray(t_src, dtype=float) + float(shift_s)
    return np.interp(
        np.asarray(t_ref, dtype=float),
        shifted_time,
        np.asarray(y_src, dtype=float),
        left=np.nan,
        right=np.nan,
    )


def fit_scale_and_shift(
    t_s: np.ndarray,
    measured_counts_zeroed: np.ndarray,
    ideal_mm: np.ndarray,
    period_s: float,
    shift_fraction: float,
    min_valid_fraction: float,
) -> dict:
    mask = np.isfinite(t_s) & np.isfinite(measured_counts_zeroed) & np.isfinite(ideal_mm)
    if mask.sum() < 10:
        raise ValueError('Need at least 10 valid samples to fit scale and time shift.')

    t_fit = np.asarray(t_s[mask], dtype=float)
    measured_fit = np.asarray(measured_counts_zeroed[mask], dtype=float)
    ideal_fit = np.asarray(ideal_mm[mask], dtype=float)

    shift_bound = abs(float(shift_fraction)) * float(period_s)
    minimum_valid_samples = max(10, int(np.ceil(min_valid_fraction * len(t_fit))))

    def objective(shift_s: float) -> float:
        shifted = shift_signal_in_time(t_fit, t_fit, measured_fit, shift_s)
        valid = np.isfinite(shifted) & np.isfinite(ideal_fit)
        if valid.sum() < minimum_valid_samples:
            return np.inf
        y = shifted[valid]
        x = ideal_fit[valid]
        measured_peak = float(np.nanmax(y))
        ideal_peak = float(np.nanmax(x))
        if measured_peak <= 0.0 or not np.isfinite(measured_peak) or not np.isfinite(ideal_peak):
            return np.inf
        scale = float(ideal_peak / measured_peak)
        residuals = scale * y - x
        return float(np.mean(residuals**2))

    opt = optimize.minimize_scalar(
        objective,
        bounds=(-shift_bound, shift_bound),
        method='bounded',
        options={'xatol': max(period_s * 1e-6, 1e-9)},
    )
    if not opt.success:
        raise RuntimeError(f'Scale/shift fit did not converge: {opt.message}')

    best_shift_s = float(opt.x)
    shifted_full = shift_signal_in_time(t_s, t_s, measured_counts_zeroed, best_shift_s)
    valid_full = np.isfinite(shifted_full) & np.isfinite(ideal_mm)
    y_full = shifted_full[valid_full]
    x_full = ideal_mm[valid_full]
    measured_peak_full = float(np.nanmax(y_full))
    ideal_peak_full = float(np.nanmax(x_full))
    if measured_peak_full <= 0.0 or not np.isfinite(measured_peak_full) or not np.isfinite(ideal_peak_full):
        raise RuntimeError('Best-fit shifted signal does not have a valid positive peak for peak-matched scaling.')

    best_scale_mm_per_count = float(ideal_peak_full / measured_peak_full)
    scaled_full_mm = best_scale_mm_per_count * shifted_full
    residuals_full_mm = scaled_full_mm - ideal_mm
    rmse_mm = float(np.sqrt(np.nanmean(residuals_full_mm[valid_full] ** 2)))

    return {
        'shift_s': best_shift_s,
        'scale_mm_per_count': best_scale_mm_per_count,
        'scaled_series_mm': scaled_full_mm,
        'shifted_counts': shifted_full,
        'residuals_mm': residuals_full_mm,
        'valid_mask': valid_full,
        'measured_peak_count': measured_peak_full,
        'ideal_peak_mm': ideal_peak_full,
        'rmse_mm': rmse_mm,
        'optimizer_result': opt,
    }

In [ ]:
df = pd.read_csv(csv_path)
_require_column(df, time_column)
_require_column(df, raw_adc_column)

raw_time = df[time_column]
raw_adc_counts = pd.to_numeric(df[raw_adc_column], errors='coerce').to_numpy(dtype=float)
time_s_full = load_time_seconds(raw_time, time_format=time_format)

window_mask = np.isfinite(time_s_full)
if analysis_start_time_s is not None:
    window_mask &= time_s_full >= float(analysis_start_time_s)
if analysis_end_time_s is not None:
    window_mask &= time_s_full <= float(analysis_end_time_s)
if not np.any(window_mask):
    raise ValueError('The requested analysis time window does not contain any valid samples.')

df_window = df.loc[window_mask].copy()
time_s = time_s_full[window_mask]
raw_adc_counts = raw_adc_counts[window_mask]

coeffs = (c0, c1, c2, c3)
corrected_counts = apply_polynomial_correction(raw_adc_counts, coeffs)
unwrapped_counts = unwrap_periodic_signal(
    corrected_counts,
    wrap_span=wrap_span_counts,
    low_band_max_count=wrap_low_band_max_count,
    high_band_min_count=wrap_high_band_min_count,
)
wrap_count = np.rint((unwrapped_counts - corrected_counts) / wrap_span_counts)
unwrapped_counts_zeroed = unwrapped_counts - np.nanmin(unwrapped_counts)

processed_df = pd.DataFrame(
    {
        'time_s': time_s,
        'raw_adc_counts': raw_adc_counts,
        'corrected_counts': corrected_counts,
        'wrap_count': wrap_count,
        'unwrapped_counts': unwrapped_counts,
        'unwrapped_counts_zeroed': unwrapped_counts_zeroed,
    }
)

display(df_window.head())
display(processed_df.head())

In [ ]:
fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=('Raw ADC signal', 'Polynomial-corrected signal', 'Unwrapped corrected signal', 'Wraps'),
)
fig.add_trace(
    go.Scatter(x=processed_df['time_s'], y=processed_df['raw_adc_counts'], mode='lines', name='Raw counts', line=dict(width=1.0)),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=processed_df['time_s'], y=processed_df['corrected_counts'], mode='lines', name='Corrected counts', line=dict(width=1.0, color='orange')),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(x=processed_df['time_s'], y=processed_df['unwrapped_counts'], mode='lines', name='Unwrapped counts', line=dict(width=1.0, color='green')),
    row=3,
    col=1,
)
fig.add_trace(
    go.Scatter(x=processed_df['time_s'], y=processed_df['wrap_count'], mode='lines', name='Wrap count', line=dict(width=1.0, color='purple')),
    row=4,
    col=1,
)
fig.update_yaxes(title_text='Raw counts', row=1, col=1)
fig.update_yaxes(title_text='Corrected counts', row=2, col=1)
fig.update_yaxes(title_text='Unwrapped counts', row=3, col=1)
fig.update_yaxes(title_text='Wraps', row=4, col=1)
fig.update_xaxes(title_text='Time [s]', row=4, col=1, rangeslider=dict(visible=True))
fig.update_layout(height=1100, hovermode='x unified', legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0.0))
fig.show()

In [ ]:
period_info = estimate_period_seconds(
    processed_df['time_s'].to_numpy(dtype=float),
    processed_df['corrected_counts'].to_numpy(dtype=float),
    wrap_span=wrap_span_counts,
    min_rpm=period_search_min_rpm,
    max_rpm=period_search_max_rpm,
    sample_rate_hz_override=sample_rate_hz_override,
)

data_estimated_period_s = period_info['period_s']
data_estimated_frequency_hz = 1.0 / data_estimated_period_s
data_estimated_speed_rpm = 60.0 * data_estimated_frequency_hz

if driving_speed_rpm_override is None:
    selected_period_source = 'data_estimate'
    estimated_speed_rpm = data_estimated_speed_rpm
    estimated_frequency_hz = data_estimated_frequency_hz
    estimated_period_s = data_estimated_period_s
else:
    estimated_speed_rpm = float(driving_speed_rpm_override)
    if not np.isfinite(estimated_speed_rpm) or estimated_speed_rpm <= 0.0:
        raise ValueError('driving_speed_rpm_override must be positive when provided.')
    selected_period_source = 'driving_speed_rpm_override'
    estimated_frequency_hz = estimated_speed_rpm / 60.0
    estimated_period_s = 1.0 / estimated_frequency_hz

instantaneous_period_info = estimate_instantaneous_period_model(
    processed_df['time_s'].to_numpy(dtype=float),
    processed_df['unwrapped_counts_zeroed'].to_numpy(dtype=float),
    nominal_period_s=estimated_period_s,
    sample_rate_hz=period_info['sample_rate_hz'],
    min_peak_spacing_fraction=instantaneous_period_min_peak_spacing_fraction,
    peak_prominence_fraction=instantaneous_period_peak_prominence_fraction,
    smoothing_fraction_of_period=instantaneous_period_smoothing_fraction_of_period,
)

cycle_periods_s = instantaneous_period_info['cycle_periods_s']
cycle_speed_rpm = instantaneous_period_info['cycle_speed_rpm']

print(f"Estimated sample rate: {period_info['sample_rate_hz']:.3f} Hz")
print(f"Data-derived period: {data_estimated_period_s:.6f} s")
print(f"Data-derived frequency: {data_estimated_frequency_hz:.6f} Hz")
print(f"Data-derived crank speed: {data_estimated_speed_rpm:.3f} RPM")
print(f"Selected timing source: {selected_period_source}")
print(f"Selected period: {estimated_period_s:.6f} s")
print(f"Selected frequency: {estimated_frequency_hz:.6f} Hz")
print(f"Selected crank speed: {estimated_speed_rpm:.3f} RPM")
print(f"Period estimate method: {period_info['method']}")
print(f"Welch cross-check frequency: {period_info['dominant_frequency_hz']:.6f} Hz")
print(f"Candidate periods [s]: {np.array2string(period_info['candidate_periods_s'], precision=6)}")
print(f"Cycle anchors detected: {len(instantaneous_period_info['anchor_times_s'])}")
print(f"Cycle-period mean/min/max: {np.mean(cycle_periods_s):.6f} / {np.min(cycle_periods_s):.6f} / {np.max(cycle_periods_s):.6f} s")
print(f"Cycle-speed mean/min/max: {np.mean(cycle_speed_rpm):.3f} / {np.max(cycle_speed_rpm):.3f} / {np.min(cycle_speed_rpm):.3f} RPM")
print(f"Cycle-speed std: {np.std(cycle_speed_rpm):.3f} RPM")

In [ ]:
fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.08,
    subplot_titles=(
        'Signal used for cycle-anchor detection',
        'Circular autocorrelation period search',
        'Cycle-to-cycle speed from measured signal',
    ),
)
fig.add_trace(
    go.Scatter(
        x=processed_df['time_s'],
        y=processed_df['unwrapped_counts_zeroed'],
        mode='lines',
        name='Unwrapped zeroed counts',
        line=dict(width=1.0, color='rgba(30, 30, 30, 0.55)'),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=instantaneous_period_info['t_valid'],
        y=instantaneous_period_info['smoothed_signal'],
        mode='lines',
        name='Smoothed anchor signal',
        line=dict(width=1.5, color='royalblue'),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=instantaneous_period_info['anchor_times_s'],
        y=instantaneous_period_info['anchor_values'],
        mode='markers',
        name='Cycle anchors',
        marker=dict(size=7, color='crimson', symbol='diamond'),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=period_info['lag_s'], y=period_info['autocorr'], mode='lines', name='Circular autocorrelation', line=dict(width=1.2)),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=period_info['lag_s'][period_info['search_mask']],
        y=period_info['autocorr'][period_info['search_mask']],
        mode='lines',
        name='Autocorr search window',
        line=dict(width=2.0, color='orange'),
    ),
    row=2,
    col=1,
)
fig.add_vline(x=period_info['period_s'], line_dash='dash', line_color='red', row=2, col=1)
fig.add_trace(
    go.Scatter(
        x=instantaneous_period_info['cycle_mid_times_s'],
        y=instantaneous_period_info['cycle_speed_rpm'],
        mode='lines+markers',
        name='Cycle speed [RPM]',
        line=dict(width=1.5, color='seagreen'),
        marker=dict(size=5),
    ),
    row=3,
    col=1,
)
fig.add_hline(y=estimated_speed_rpm, line_dash='dash', line_color='gray', row=3, col=1)
fig.update_yaxes(title_text='Counts', row=1, col=1)
fig.update_yaxes(title_text='Autocorrelation', row=2, col=1)
fig.update_yaxes(title_text='Speed [RPM]', row=3, col=1)
fig.update_xaxes(title_text='Time [s]', row=1, col=1, rangeslider=dict(visible=True))
fig.update_xaxes(title_text='Lag [s]', row=2, col=1, rangeslider=dict(visible=True))
fig.update_xaxes(title_text='Cycle midpoint time [s]', row=3, col=1, rangeslider=dict(visible=True))
fig.update_layout(height=1050, hovermode='x unified', legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0.0))
fig.show()

In [ ]:
constant_speed_ideal_displacement_mm = ideal_profile_from_time(
    processed_df['time_s'].to_numpy(dtype=float),
    period_s=estimated_period_s,
    r_mm=crank_pin_offset_r_mm,
    l_mm=entry_offset_l_mm,
)

instantaneous_ideal_displacement_mm = ideal_profile_from_phase(
    instantaneous_period_info['phase_rad'],
    r_mm=crank_pin_offset_r_mm,
    l_mm=entry_offset_l_mm,
)

constant_speed_fit_info = fit_scale_and_shift(
    processed_df['time_s'].to_numpy(dtype=float),
    processed_df['unwrapped_counts_zeroed'].to_numpy(dtype=float),
    constant_speed_ideal_displacement_mm,
    period_s=estimated_period_s,
    shift_fraction=fit_shift_search_fraction_of_period,
    min_valid_fraction=minimum_valid_fraction_for_fit,
)

fit_info = fit_scale_and_shift(
    processed_df['time_s'].to_numpy(dtype=float),
    processed_df['unwrapped_counts_zeroed'].to_numpy(dtype=float),
    instantaneous_ideal_displacement_mm,
    period_s=estimated_period_s,
    shift_fraction=fit_shift_search_fraction_of_period,
    min_valid_fraction=minimum_valid_fraction_for_fit,
)

analysis_df = processed_df.copy()
analysis_df['ideal_displacement_constant_mm'] = constant_speed_ideal_displacement_mm
analysis_df['ideal_displacement_mm'] = instantaneous_ideal_displacement_mm
analysis_df['instantaneous_phase_rad'] = instantaneous_period_info['phase_rad']
analysis_df['instantaneous_period_s'] = instantaneous_period_info['instantaneous_period_s']
analysis_df['instantaneous_speed_rpm'] = np.where(
    np.isfinite(analysis_df['instantaneous_period_s']) & (analysis_df['instantaneous_period_s'] > 0.0),
    60.0 / analysis_df['instantaneous_period_s'],
    np.nan,
)
analysis_df['shifted_unwrapped_counts'] = fit_info['shifted_counts']
analysis_df['scaled_measured_mm'] = fit_info['scaled_series_mm']
analysis_df['residual_mm'] = fit_info['residuals_mm']
analysis_df['constant_speed_residual_mm'] = constant_speed_fit_info['residuals_mm']
analysis_df['fit_valid'] = fit_info['valid_mask']
analysis_df['constant_speed_fit_valid'] = constant_speed_fit_info['valid_mask']

fit_info['analysis_df'] = analysis_df
fit_info['constant_speed_fit_info'] = constant_speed_fit_info

print(f"Constant-speed RMSE: {constant_speed_fit_info['rmse_mm']:.6f} mm")
print(f"Variable-speed RMSE: {fit_info['rmse_mm']:.6f} mm")
print(f"RMSE improvement: {constant_speed_fit_info['rmse_mm'] - fit_info['rmse_mm']:.6f} mm")
print(f"Best-fit scale: {fit_info['scale_mm_per_count']:.9f} mm/count")
print(f"Best-fit time shift: {fit_info['shift_s']:.6f} s")

In [ ]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=(
        'Measured displacement against constant-speed and variable-speed ideals',
        'Residuals: constant-speed baseline vs variable-speed model',
    ),
)
fig.add_trace(
    go.Scatter(
        x=analysis_df['time_s'],
        y=analysis_df['ideal_displacement_constant_mm'],
        mode='lines',
        name='Constant-speed ideal [mm]',
        line=dict(width=1.2, dash='dot', color='gray'),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=analysis_df['time_s'],
        y=analysis_df['ideal_displacement_mm'],
        mode='lines',
        name='Variable-speed ideal [mm]',
        line=dict(width=2.0, color='royalblue'),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=analysis_df['time_s'],
        y=analysis_df['scaled_measured_mm'],
        mode='lines',
        name='Scaled measured series [mm]',
        line=dict(width=1.2, color='black'),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=analysis_df['time_s'],
        y=analysis_df['constant_speed_residual_mm'],
        mode='lines',
        name='Constant-speed residual [mm]',
        line=dict(width=1.0, color='rgba(120, 120, 120, 0.8)'),
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=analysis_df['time_s'],
        y=analysis_df['residual_mm'],
        mode='lines',
        name='Variable-speed residual [mm]',
        line=dict(width=1.0, color='red'),
    ),
    row=2,
    col=1,
)
fig.add_hline(y=0.0, line_width=0.8, line_color='black', row=2, col=1)
fig.update_yaxes(title_text='Displacement [mm]', row=1, col=1)
fig.update_yaxes(title_text='Residual [mm]', row=2, col=1)
fig.update_xaxes(title_text='Time [s]', row=2, col=1, rangeslider=dict(visible=True))
fig.update_layout(height=820, hovermode='x unified', legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0.0))
fig.show()

In [ ]:
valid_residuals = analysis_df.loc[analysis_df['fit_valid'], 'residual_mm'].to_numpy(dtype=float)
valid_constant_residuals = analysis_df.loc[analysis_df['constant_speed_fit_valid'], 'constant_speed_residual_mm'].to_numpy(dtype=float)
cycle_periods_s = instantaneous_period_info['cycle_periods_s']
cycle_speed_rpm = instantaneous_period_info['cycle_speed_rpm']
constant_speed_rmse_mm = float(np.sqrt(np.nanmean(valid_constant_residuals**2)))
variable_speed_rmse_mm = float(np.sqrt(np.nanmean(valid_residuals**2)))

summary = {
    'csv_path': str(csv_path),
    'time_column': time_column,
    'raw_adc_column': raw_adc_column,
    'analysis_start_time_s': None if analysis_start_time_s is None else float(analysis_start_time_s),
    'analysis_end_time_s': None if analysis_end_time_s is None else float(analysis_end_time_s),
    'driving_speed_rpm_override': None if driving_speed_rpm_override is None else float(driving_speed_rpm_override),
    'selected_period_source': selected_period_source,
    'instantaneous_model_source': 'cycle_peak_intervals',
    'wrap_span_counts': float(wrap_span_counts),
    'wrap_low_band_max_count': float(wrap_low_band_max_count),
    'wrap_high_band_min_count': float(wrap_high_band_min_count),
    'poly_c0': float(c0),
    'poly_c1': float(c1),
    'poly_c2': float(c2),
    'poly_c3': float(c3),
    'geometry_r_mm': float(crank_pin_offset_r_mm),
    'geometry_l_mm': float(entry_offset_l_mm),
    'sample_count': int(len(analysis_df)),
    'valid_fit_samples': int(analysis_df['fit_valid'].sum()),
    'duration_s': float(np.nanmax(analysis_df['time_s']) - np.nanmin(analysis_df['time_s'])),
    'estimated_sample_rate_hz': float(period_info['sample_rate_hz']),
    'data_estimated_period_s': float(data_estimated_period_s),
    'data_estimated_frequency_hz': float(data_estimated_frequency_hz),
    'data_estimated_speed_rpm': float(data_estimated_speed_rpm),
    'estimated_period_s': float(estimated_period_s),
    'estimated_frequency_hz': float(estimated_frequency_hz),
    'estimated_speed_rpm': float(estimated_speed_rpm),
    'cycle_anchor_count': int(len(instantaneous_period_info['anchor_times_s'])),
    'cycle_period_mean_s': float(np.nanmean(cycle_periods_s)),
    'cycle_period_std_s': float(np.nanstd(cycle_periods_s)),
    'cycle_period_min_s': float(np.nanmin(cycle_periods_s)),
    'cycle_period_max_s': float(np.nanmax(cycle_periods_s)),
    'cycle_speed_mean_rpm': float(np.nanmean(cycle_speed_rpm)),
    'cycle_speed_std_rpm': float(np.nanstd(cycle_speed_rpm)),
    'cycle_speed_min_rpm': float(np.nanmin(cycle_speed_rpm)),
    'cycle_speed_max_rpm': float(np.nanmax(cycle_speed_rpm)),
    'fit_scale_mm_per_count': float(fit_info['scale_mm_per_count']),
    'fit_time_shift_s': float(fit_info['shift_s']),
    'peak_matched_measured_peak_count': float(fit_info['measured_peak_count']),
    'peak_matched_ideal_peak_mm': float(fit_info['ideal_peak_mm']),
    'peak_matched_scaled_peak_mm': float(np.nanmax(analysis_df.loc[analysis_df['fit_valid'], 'scaled_measured_mm'].to_numpy(dtype=float))),
    'constant_speed_rmse_mm': constant_speed_rmse_mm,
    'rmse_mm': variable_speed_rmse_mm,
    'rmse_improvement_mm': float(constant_speed_rmse_mm - variable_speed_rmse_mm),
    'constant_speed_mae_mm': float(np.nanmean(np.abs(valid_constant_residuals))),
    'mae_mm': float(np.nanmean(np.abs(valid_residuals))),
    'max_abs_residual_mm': float(np.nanmax(np.abs(valid_residuals))),
    'raw_min_count': float(np.nanmin(analysis_df['raw_adc_counts'])),
    'raw_max_count': float(np.nanmax(analysis_df['raw_adc_counts'])),
    'unwrapped_min_count': float(np.nanmin(analysis_df['unwrapped_counts'])),
    'unwrapped_max_count': float(np.nanmax(analysis_df['unwrapped_counts'])),
    'ideal_min_mm': float(np.nanmin(analysis_df['ideal_displacement_mm'])),
    'ideal_max_mm': float(np.nanmax(analysis_df['ideal_displacement_mm'])),
}

summary_df = pd.DataFrame({'metric': list(summary.keys()), 'value': list(summary.values())})
display(summary_df)

## Notes

- The polynomial is applied directly to the raw ADC counts and returns corrected counts.
- `analysis_start_time_s` and `analysis_end_time_s` define the inclusive time window, in seconds from the start of the file, used by the notebook.
- `driving_speed_rpm_override` is optional. Leave it as `None` to use the notebook's current data-driven period estimate; set it to a positive RPM value to force the ideal-profile timing used for fitting.
- Unwrapping now uses the sign of the recent travel direction together with configurable low and high wrap bands. A wrap is only declared when the signal crosses from the high band to the low band while moving upward, or from the low band to the high band while moving downward.
- The fit uses the measured series shifted in time and scaled in amplitude only. Before fitting, the unwrapped counts are zeroed to their minimum so the measured series can be compared against the ideal displacement profile, which is also referenced to its minimum string length. The scale is peak-matched so the fitted measured peak displacement equals the ideal peak displacement exactly over the valid fit window.
- Period estimation is based on circular autocorrelation of the wrapped corrected counts, with an RPM-bounded search window and a Welch-spectrum cross-check.